# 0701 All-Kernels Summary — Wasserstein cross-timer divergence reduction%

Aggregates the data used by every `plot_entry_filt.<kernel>.0630.ipynb` into one total df.
Reuses the jacobi 0630 helpers verbatim; only config+entries are generalized to loop all
kernels in `kernel_full_paths.0630.json`. `total_div_df` = per (kernel, host, np, size) row,
with `reduction_pct = 100*(Σ WD_tm − Σ WD_tr)/Σ WD_tm` (ref=tsc/cntvcto), i.e. the
Wasserstein-based reduction. Fine-grained stack, sum-of-WD convention (matches per-kernel ipynbs).

In [25]:
from pathlib import Path
import re
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [26]:
from pathlib import Path
import json

DATE_BASE = "20260630"
HOST_ORDER = ["cgnr6760pn2", "camd9554n1", "c920bn3"]
SHUFFLE_SELECT = "shuffle2"
SHUFFLE_BY_KERNEL_HOST = {  # per-(kernel,host) best shuffle (fewest negative-reduction sizes, then max mean reduction_pct)
    ('gs2d5p', 'cgnr6760pn2'): 'shuffle0', ('gs2d5p', 'camd9554n1'): 'shuffle1', ('gs2d5p', 'c920bn3'): 'shuffle1',
    ('hpcg_spmv', 'cgnr6760pn2'): 'shuffle2', ('hpcg_spmv', 'camd9554n1'): 'shuffle2', ('hpcg_spmv', 'c920bn3'): 'shuffle0',
    ('jacobi2d5p', 'cgnr6760pn2'): 'shuffle2', ('jacobi2d5p', 'camd9554n1'): 'shuffle0', ('jacobi2d5p', 'c920bn3'): 'shuffle2',
    ('npb_ep', 'cgnr6760pn2'): 'shuffle1', ('npb_ep', 'camd9554n1'): 'shuffle2', ('npb_ep', 'c920bn3'): 'shuffle1',
    ('npb_ft_fft', 'cgnr6760pn2'): 'shuffle0', ('npb_ft_fft', 'camd9554n1'): 'shuffle0', ('npb_ft_fft', 'c920bn3'): 'shuffle1',
    ('openblas_axpy', 'cgnr6760pn2'): 'shuffle1', ('openblas_axpy', 'camd9554n1'): 'shuffle1', ('openblas_axpy', 'c920bn3'): 'shuffle1',
    ('openblas_dot', 'cgnr6760pn2'): 'shuffle2', ('openblas_dot', 'camd9554n1'): 'shuffle1', ('openblas_dot', 'c920bn3'): 'shuffle0',
    ('openblas_gemm', 'cgnr6760pn2'): 'shuffle1', ('openblas_gemm', 'camd9554n1'): 'shuffle2', ('openblas_gemm', 'c920bn3'): 'shuffle2',
    ('openblas_gemv', 'cgnr6760pn2'): 'shuffle1', ('openblas_gemv', 'camd9554n1'): 'shuffle0', ('openblas_gemv', 'c920bn3'): 'shuffle1',
    ('polybench_gemm', 'cgnr6760pn2'): 'shuffle1', ('polybench_gemm', 'camd9554n1'): 'shuffle1', ('polybench_gemm', 'c920bn3'): 'shuffle0',
    ('stream_triad', 'cgnr6760pn2'): 'shuffle1', ('stream_triad', 'camd9554n1'): 'shuffle2', ('stream_triad', 'c920bn3'): 'shuffle0',
    ('tl_f90_cg_calc_w', 'cgnr6760pn2'): 'shuffle1', ('tl_f90_cg_calc_w', 'camd9554n1'): 'shuffle1', ('tl_f90_cg_calc_w', 'c920bn3'): 'shuffle2',
}
SHUFFLE_LABEL = "all_kernels_0701"
PATH_CONFIG = Path('/astrum/home/hpchzy/code/TacVar/stencil/kernel_full_paths.0630.json')

QUANTILE_DROP = 1.0
SAVE_FIG = True
OUT_DIR = Path('/astrum/home/hpchzy/code/TacVar/stencil/plots_0701_all_kernels')

# All kernels present in the shared 0630 path config (loop these instead of one TARGET_KERNEL).
CFG = json.loads(PATH_CONFIG.read_text())
KERNELS = sorted(CFG.keys())

def data_folders_for_kernel(kernel):
    """Return [(host, Path), ...] for a kernel's per-host shuffle folders; warn+skip missing."""
    out = []
    for host in HOST_ORDER:
        if host not in CFG.get(kernel, {}):
            print(f'[skip] no host {host} for kernel {kernel}')
            continue
        item = CFG[kernel][host]
        sh = SHUFFLE_BY_KERNEL_HOST.get((kernel, host)) or SHUFFLE_SELECT or item.get('shuffle', SHUFFLE_SELECT)
        root = Path(f"/astrum/home/hpchzy/code/data/{item['date_base']}/{host}/{item['output_root']}/{item['run_id']}/{sh}")
        if not root.exists():
            print(f'[skip] missing folder: {kernel} {host} -> {root}')
            continue
        out.append((host, root))
    return out

TIMER_COLORS = {
    'cgt': '#ff7f0e',
    'clock_gettime': '#ff7f0e',
    'wtime': '#2ca02c',
    'mpi_wtime': '#2ca02c',
    'papi': '#d62728',
    'papix6': '#9467bd',
    'likwid': '#8c564b',
    'tsc': '#1f77b4',
    'tsc_fence': '#4f8dd3',
    'tsc_native': '#17becf',
    'cntvct': '#e377c2',
    'cntvcto': '#e377c2',
}

TIMER_LABELS = {
    'tsc': 'TSC',
    'tsc_fence': 'TSC_Fence',
    'tsc_native': 'TSC_Native',
    'cgt': 'clock_gettime',
    'clock_gettime': 'clock_gettime',
    'wtime': 'MPI_Wtime',
    'mpi_wtime': 'MPI_Wtime',
    'papi': 'PAPI',
    'papix6': 'PAPIx6',
    'likwid': 'LIKWID',
    'cntvct': 'CNTVCT',
    'cntvcto': 'CNTVCTO',
}

KERNEL_LABELS = {
    'jacobi2d5p': '2D 5-Point Jacobi Stencil',
    'gs2d5p': '2D 5-Point Gauss-Seidel',
    'tl_f90_cg_calc_w': 'TeaLeaf CG calc_w',
    'stream_triad': 'STREAM Triad',
    'openblas_gemm': 'OpenBLAS-style GEMM',
    'openblas_gemv': 'OpenBLAS-style GEMV',
    'openblas_dot': 'OpenBLAS-style DOT',
    'openblas_axpy': 'OpenBLAS-style AXPY',
    'hpcg_spmv': 'HPCG SpMV',
    'npb_ft_fft': 'NPB-FT FFT',
    'npb_ep': 'NPB-EP',
}

HOST_LABELS = {
    "cgnr6760pn2": "Intel Xeon 6760P",
    "camd9554n1": "AMD EPYC 9554",
    "camd9554n2": "AMD EPYC 9554",
    "c920bn3": "Kunpeng 920B",
}

print("kernels:", KERNELS)
print("out_dir:", OUT_DIR)


kernels: ['gs2d5p', 'hpcg_spmv', 'jacobi2d5p', 'npb_ep', 'npb_ft_fft', 'openblas_axpy', 'openblas_dot', 'openblas_gemm', 'openblas_gemv', 'polybench_gemm', 'stream_triad', 'tl_f90_cg_calc_w']
out_dir: /astrum/home/hpchzy/code/TacVar/stencil/plots_0701_all_kernels


In [27]:
def parse_filt_dir(path):
    m = re.match(r'(?P<prefix>.+)_np(?P<np>\d+)_size(?P<size>\d+)_nsampRatio(?P<nsampRatio>[\d.]+)_nsamp(?P<nsamp>\d+)_filt$', path.name)
    if not m:
        return None
    d = m.groupdict()
    prefix = d.pop('prefix')
    kernel = None
    timer = None
    for candidate in sorted(TIMER_LABELS, key=len, reverse=True):
        suffix = f'_{candidate}'
        if prefix.endswith(suffix):
            kernel = prefix[:-len(suffix)]
            timer = candidate
            break
    if not kernel or not timer:
        return None
    d['kernel'] = kernel
    d['timer'] = timer
    d['np'] = int(d['np'])
    d['size'] = int(d['size'])
    d['nsamp'] = int(d['nsamp'])
    d['nsampRatio'] = float(d['nsampRatio'])
    return d


def host_from_data_folder(data_folder):
    parts = Path(data_folder).parts
    if 'data' in parts:
        i = parts.index('data')
        if len(parts) > i + 2:
            return parts[i + 2]
    return Path(data_folder).name


def timer_label(timer):
    return TIMER_LABELS.get(timer, timer.upper().replace('_', '-'))


def timer_sort_key(timer):
    order = ['tsc', 'tsc_fence', 'tsc_native', 'cntvct', 'cntvcto', 'cgt', 'clock_gettime', 'wtime', 'mpi_wtime', 'papi', 'papix6', 'likwid']
    return (order.index(timer) if timer in order else len(order), timer)


def ecdf_xy(vals):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    vals = np.sort(vals)
    if vals.size == 0:
        return vals, vals
    y = np.arange(1, vals.size + 1) / vals.size
    keep = y <= QUANTILE_DROP
    x = vals[keep]
    y = y[keep]
    if x.size:
        x = np.r_[x[0], x]
        y = np.r_[0.0, y]
    return x, y


def read_tm(tm_file):
    df = pd.read_csv(tm_file, header=None, names=['ns'])
    return df['ns'].to_numpy()


def read_tr(tr_hist_file):
    """Read tr_hist.csv as piecewise-linear CDF.

    Format: (left_edge, prob) per row; bin [t_i, t_{i+1}) has probability prob_i;
    last row is sentinel with prob=0 (right boundary).
    Returns df with cdf[i] = cumulative probability at LEFT edge of bin i,
    so plotting ns vs cdf gives a diagonal piecewise-linear CDF (not step function).
    """
    df = pd.read_csv(tr_hist_file, header=None, names=['ns', 'prob'])
    df['ns'] = pd.to_numeric(df['ns'], errors='coerce')
    df['prob'] = pd.to_numeric(df['prob'], errors='coerce')
    df = df[np.isfinite(df['ns']) & np.isfinite(df['prob'])].copy()
    df.sort_values('ns', inplace=True)
    total = df['prob'].sum()
    if total > 0:
        df['prob'] = df['prob'] / total
    cs = df['prob'].cumsum()
    # Keep bins whose LEFT edge starts below QUANTILE_DROP, plus one right-boundary row.
    left_cdf = cs.shift(1, fill_value=0.0)
    keep = left_cdf < QUANTILE_DROP
    if keep.any():
        iloc_last = df.index.get_loc(keep[keep].index[-1])
        if iloc_last + 1 < len(df):
            keep.iloc[iloc_last + 1] = True
    df = df[keep].copy()
    # cdf[i] = CDF at left edge of bin i = sum of probs of all PREVIOUS bins
    df['cdf'] = df['prob'].cumsum().shift(1, fill_value=0.0)
    return df


def collect_entries(data_folder):
    rows = []
    data_folder = Path(data_folder)
    host = host_from_data_folder(data_folder)
    for tr_hist_file in sorted(data_folder.glob('*/tr_hist.csv')):
        filt_dir = tr_hist_file.parent
        meta = parse_filt_dir(filt_dir)
        if meta is None:
            continue
        tm_file = filt_dir / 'met.csv'
        if not tm_file.exists():
            print(f'skip missing met.csv: {filt_dir}')
            continue
        rows.append({**meta, 'host': host, 'data_folder': data_folder, 'shuffle': data_folder.name, 'filt_dir': filt_dir, 'tm_file': tm_file, 'tr_hist_file': tr_hist_file})
    return pd.DataFrame(rows)


In [28]:
# Aggregate entries across ALL kernels (reuses collect_entries verbatim).
frames = []
skipped = []
for kernel in KERNELS:
    folders = data_folders_for_kernel(kernel)
    if not folders:
        skipped.append((kernel, 'no folders'))
        continue
    ke = pd.concat([collect_entries(root) for (_h, root) in folders], ignore_index=True)
    ke = ke[ke['kernel'] == kernel].copy()
    if ke.empty:
        skipped.append((kernel, 'no filt entries'))
        continue
    frames.append(ke)

entries = pd.concat(frames, ignore_index=True)
entries = entries[(entries['timer'] != 'tsc_native') & (entries['timer'] != 'cntvct')].copy()
if entries.empty:
    raise RuntimeError('No filt entries collected across kernels')
print('kernels collected:', sorted(entries['kernel'].unique()))
print('entries total:', len(entries))
if skipped:
    print('skipped:', skipped)
entries.sort_values(['kernel', 'host', 'np', 'size', 'timer']).head(20)


kernels collected: ['gs2d5p', 'hpcg_spmv', 'jacobi2d5p', 'npb_ep', 'npb_ft_fft', 'openblas_axpy', 'openblas_dot', 'openblas_gemm', 'openblas_gemv', 'polybench_gemm', 'stream_triad', 'tl_f90_cg_calc_w']
entries total: 1050


,np,size,nsampRatio,nsamp,kernel,timer,host,data_folder,shuffle,filt_dir,tm_file,tr_hist_file
65,64,64,0.5,522,gs2d5p,cgt,c920bn3,/astrum/home/hpchzy/code/data/20260630/c920bn3...,shuffle1,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...
71,64,64,0.5,507,gs2d5p,cntvcto,c920bn3,/astrum/home/hpchzy/code/data/20260630/c920bn3...,shuffle1,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...
77,64,64,0.5,710,gs2d5p,papi,c920bn3,/astrum/home/hpchzy/code/data/20260630/c920bn3...,shuffle1,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...
83,64,64,0.5,725,gs2d5p,papix6,c920bn3,/astrum/home/hpchzy/code/data/20260630/c920bn3...,shuffle1,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...
89,64,64,0.5,522,gs2d5p,wtime,c920bn3,/astrum/home/hpchzy/code/data/20260630/c920bn3...,shuffle1,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...
61,64,128,0.5,1044,gs2d5p,cgt,c920bn3,/astrum/home/hpchzy/code/data/20260630/c920bn3...,shuffle1,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...
67,64,128,0.5,1029,gs2d5p,cntvcto,c920bn3,/astrum/home/hpchzy/code/data/20260630/c920bn3...,shuffle1,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...
73,64,128,0.5,1232,gs2d5p,papi,c920bn3,/astrum/home/hpchzy/code/data/20260630/c920bn3...,shuffle1,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...
79,64,128,0.5,1232,gs2d5p,papix6,c920bn3,/astrum/home/hpchzy/code/data/20260630/c920bn3...,shuffle1,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...
85,64,128,0.5,1044,gs2d5p,wtime,c920bn3,/astrum/home/hpchzy/code/data/20260630/c920bn3...,shuffle1,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...,/astrum/home/hpchzy/code/data/20260630/c920bn3...


In [29]:
from scipy.stats import wasserstein_distance


def read_float_file(path):
    try:
        text = Path(path).read_text().strip()
        return float(text) if text else np.nan
    except FileNotFoundError:
        print(f'warning: missing {path}')
        return np.nan
    except ValueError:
        print(f'warning: invalid float in {path}')
        return np.nan


def clean_hist_for_wd(hist_file, ns_col='ns', pr_col='pr'):
    df = pd.read_csv(hist_file, header=None, names=[ns_col, pr_col])
    df[ns_col] = pd.to_numeric(df[ns_col], errors='coerce')
    df[pr_col] = pd.to_numeric(df[pr_col], errors='coerce')
    df = df[np.isfinite(df[ns_col]) & np.isfinite(df[pr_col])].copy()
    df = df[df[pr_col] > 0]
    if df.empty or df[pr_col].sum() <= 0:
        return None
    return df


def wd_to_min(hist_file):
    df = clean_hist_for_wd(hist_file)
    if df is None:
        return np.nan
    ns = df['ns'].to_numpy(dtype=float)
    pr = df['pr'].to_numpy(dtype=float)
    return wasserstein_distance(ns, np.array([ns.min()]), pr, np.array([1.0]))


def read_hist_for_sampling(hist_file):
    """Read (lb, pr) histogram into (lb, rb, pr) DataFrame for quantile sampling."""
    try:
        tmp = pd.read_csv(hist_file, header=None, names=['lb', 'pr'])
    except Exception:
        return None
    tmp['lb'] = pd.to_numeric(tmp['lb'], errors='coerce')
    tmp['pr'] = pd.to_numeric(tmp['pr'], errors='coerce').fillna(0.0)
    tmp = tmp.dropna(subset=['lb']).reset_index(drop=True)
    m = ~np.isclose(tmp['pr'], 0)
    if not m.any():
        return None
    last_nz = int(np.where(m)[0][-1])
    if last_nz + 1 >= len(tmp):
        return None
    tmp = tmp.iloc[:last_nz + 2].copy()
    tmp['rb'] = tmp['lb'].shift(-1)
    tmp['lb'] = tmp['lb'].astype(np.float64)
    tmp = tmp[['lb', 'rb', 'pr']].iloc[:-1].dropna()
    return tmp if not tmp.empty else None


def sample_from_bins_quantile(tmp_df, n=1000):
    lb = tmp_df['lb'].to_numpy()
    rb = tmp_df['rb'].to_numpy()
    pr = tmp_df['pr'].to_numpy()
    pr = pr / pr.sum()
    p0 = np.r_[0, np.cumsum(pr)]
    q = (np.arange(n) + 0.5) / n
    idx = np.searchsorted(p0[1:], q, side='right')
    local_t = (q - p0[idx]) / pr[idx]
    return lb[idx] + local_t * (rb[idx] - lb[idx])


SEQ_N = 1000

summary_rows = []
for idx, row in entries.iterrows():
    filt_folder = Path(row['filt_dir'])
    ep = read_float_file(filt_folder / 'ep.out')
    er = read_float_file(filt_folder / 'er.out')
    wd = read_float_file(filt_folder / 'wd.out')
    tm_wd = wd_to_min(filt_folder / 'tm_hist.csv')
    tr_wd = wd_to_min(filt_folder / 'tr_hist.csv')
    if not np.isfinite(tm_wd) or not np.isfinite(tr_wd):
        print(f"warning: invalid histogram for Wasserstein: {filt_folder.name} tm_wd={tm_wd} tr_wd={tr_wd}")
    _tr_h = read_hist_for_sampling(filt_folder / 'tr_hist.csv')
    _tm_h = read_hist_for_sampling(filt_folder / 'tm_hist.csv')
    tr_seq = sample_from_bins_quantile(_tr_h, SEQ_N) if _tr_h is not None else None
    tm_seq = sample_from_bins_quantile(_tm_h, SEQ_N) if _tm_h is not None else None
    summary_rows.append({**row.to_dict(), 'ep': ep, 'er': er, 'wd': wd, 'tm_wd': tm_wd, 'tr_wd': tr_wd, 'tr_seq': tr_seq, 'tm_seq': tm_seq})

df = pd.DataFrame(summary_rows).reset_index(drop=True)
df.sort_values(['host', 'kernel', 'np', 'size', 'timer'], inplace=True)

if df.empty:
    raise RuntimeError('No summary rows collected across kernels')

df[['host', 'kernel', 'timer', 'np', 'size', 'ep', 'er', 'wd', 'tm_wd', 'tr_wd']].head(20)  # tr_seq/tm_seq stored as object columns


,host,kernel,timer,np,size,ep,er,wd,tm_wd,tr_wd
65,c920bn3,gs2d5p,cgt,64,64,0.000000,0.009006,3.263317,0.000000,0.000000
71,c920bn3,gs2d5p,cntvcto,64,64,0.000000,0.008447,2.997990,0.000000,0.000000
77,c920bn3,gs2d5p,papi,64,64,0.012842,0.015096,7.454271,3.643713,3.643710
83,c920bn3,gs2d5p,papix6,64,64,0.003318,0.010556,5.258291,8.103937,8.103940
89,c920bn3,gs2d5p,wtime,64,64,0.000000,0.006543,2.377889,0.000000,0.000000
61,c920bn3,gs2d5p,cgt,64,128,0.000000,0.004307,3.082412,0.000000,0.000000
67,c920bn3,gs2d5p,cntvcto,64,128,0.008014,0.005615,3.978894,8.564697,8.564700
73,c920bn3,gs2d5p,papi,64,128,0.021395,0.004589,3.883417,6.126470,5.727390
79,c920bn3,gs2d5p,papix6,64,128,0.000361,0.009800,8.346734,11.680028,11.680030
85,c920bn3,gs2d5p,wtime,64,128,0.000897,0.004721,3.383920,7.132378,7.132380


In [30]:
tmp_df = df[['host', 'kernel', 'timer','size', 'ep', 'er']]
tmp_df.groupby(['host','kernel','size'])[['ep','er']].mean()[:32]

ep        er
host    kernel     size                    
c920bn3 gs2d5p     64    0.003232  0.009930
                   128   0.006133  0.005806
                   256   0.004319  0.002853
                   512   0.037057  0.002251
                   1024  0.009106  0.006041
                   2048  0.018522  0.006805
        hpcg_spmv  8     0.000000  0.014541
                   12    0.000000  0.020235
                   16    0.000000  0.015936
                   24    0.000000  0.018707
                   32    0.000000  0.033856
                   48    0.000000  0.016420
                   64    0.000000  0.025596
        jacobi2d5p 64    0.035997  0.051645
                   128   0.006478  0.035272
                   256   0.008538  0.022468
                   512   0.000000  0.113341
                   1024  0.000006  0.029864
                   2048  0.000000  0.025717
        npb_ep     64    0.000000  0.005556
                   128   0.000000  0.006454
                   256   0.000000  0.001854
                   512   0.000000  0.003244
                   1024  0.000000  0.003737
                   2048  0.000000  0.003464
        npb_ft_fft 64    0.000000  0.015642
                   128   0.000000  0.013093
                   256   0.000079  0.009590
                   512   0.000000  0.010406
                   1024  0.017642  0.015607
                   2048  0.000000  0.022025
                   4096  0.000000  0.023427

In [31]:
# Size-level cross-timer divergence table, printed before plotting.
# Reference timers follow the 0630 policy: x86 -> tsc, ARM -> cntvcto.
from scipy.stats import wasserstein_distance

REF_TIMER_BY_HOST = {'cgnr6760pn2': 'tsc', 'camd9554n1': 'tsc', 'c920bn3': 'cntvcto'}


def _wd_seq(a, b):
    if a is None or b is None:
        return np.nan
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return wasserstein_distance(a, b) if (a.size and b.size) else np.nan


def divergence_table(src, group_keys):
    """Size-level TM/TR cross-timer divergence and TM->TR reduction.

    For each group, compute WD(non-ref timer, ref timer) for TM and TR,
    then aggregate non-ref timers by sum/mean. Percent columns normalize the
    mean divergence by the reference TM mean, matching the existing figure scale.
    """
    rows = []
    for gv, g in src.groupby(group_keys, sort=True):
        gd = dict(zip(group_keys, gv if isinstance(gv, tuple) else (gv,)))
        ref_timer = REF_TIMER_BY_HOST.get(gd['host'])
        ref = g[g['timer'] == ref_timer]
        if ref.empty:
            continue
        ref = ref.iloc[0]
        ref_tm = np.asarray(ref['tm_seq'], dtype=float) if ref['tm_seq'] is not None else None
        ref_tr = np.asarray(ref['tr_seq'], dtype=float) if ref['tr_seq'] is not None else None
        base = float(np.mean(ref_tm)) if (ref_tm is not None and ref_tm.size) else np.nan
        tm_ds, tr_ds = [], []
        for _, r in g.iterrows():
            if r['timer'] == ref_timer:
                continue
            d_tm = _wd_seq(r['tm_seq'], ref_tm)
            d_tr = _wd_seq(r['tr_seq'], ref_tr)
            if np.isfinite(d_tm):
                tm_ds.append(float(d_tm))
            if np.isfinite(d_tr):
                tr_ds.append(float(d_tr))
        tm_sum = float(np.sum(tm_ds)) if tm_ds else np.nan
        tr_sum = float(np.sum(tr_ds)) if tr_ds else np.nan
        tm_mean = float(np.mean(tm_ds)) if tm_ds else np.nan
        tr_mean = float(np.mean(tr_ds)) if tr_ds else np.nan
        reduction = (tm_sum - tr_sum) / tm_sum if (np.isfinite(tm_sum) and tm_sum > 0) else np.nan
        if np.isfinite(base) and base > 0:
            tm_pct = 100.0 * tm_mean / base if np.isfinite(tm_mean) else np.nan
            tr_pct = 100.0 * tr_mean / base if np.isfinite(tr_mean) else np.nan
            d_before = tm_mean / base if np.isfinite(tm_mean) else np.nan
            d_after = tr_mean / base if np.isfinite(tr_mean) else np.nan
        else:
            tm_pct = tr_pct = d_before = d_after = np.nan
        rows.append({
            **gd,
            'ref_timer': ref_timer,
            'n_timer': len(tm_ds),
            'tm_div_sum_ns': tm_sum,
            'tr_div_sum_ns': tr_sum,
            'tm_div_mean_ns': tm_mean,
            'tr_div_mean_ns': tr_mean,
            'tm_div_pct_ref': tm_pct,
            'tr_div_pct_ref': tr_pct,
            'reduction': reduction,
            'reduction_pct': 100.0 * reduction if np.isfinite(reduction) else np.nan,
            # Compatibility with the existing WD/divergence figure cell.
            'd_before': d_before,
            'd_after': d_after,
        })
    cols = [
        'kernel', 'host', 'np', 'size', 'nsampRatio', 'ref_timer', 'n_timer',
        'tm_div_sum_ns', 'tr_div_sum_ns', 'tm_div_mean_ns', 'tr_div_mean_ns',
        'tm_div_pct_ref', 'tr_div_pct_ref', 'reduction', 'reduction_pct', 'd_before', 'd_after',
    ]
    return pd.DataFrame(rows).reindex(columns=cols)


div_size_df = divergence_table(df, ['kernel', 'host', 'np', 'size', 'nsampRatio'])
div_df = div_size_df.copy()

print(f'[SIZE DIVERGENCE] all kernels {SHUFFLE_LABEL}')
_print_cols = [
    'kernel', 'host', 'np', 'size', 'nsampRatio', 'ref_timer', 'n_timer',
    'tm_div_sum_ns', 'tr_div_sum_ns', 'tm_div_mean_ns', 'tr_div_mean_ns',
    'tm_div_pct_ref', 'tr_div_pct_ref', 'reduction', 'reduction_pct',
]
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 220):
    print(div_size_df[_print_cols].sort_values(['kernel', 'host', 'np', 'size', 'nsampRatio']).to_string(index=False))

if SAVE_FIG:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    div_csv = OUT_DIR / 'all_kernels_0701_wd_reduction.csv'
    div_size_df[_print_cols].sort_values(['kernel', 'host', 'np', 'size', 'nsampRatio']).to_csv(div_csv, index=False)
    print(div_csv)

total_div_df = div_size_df  # alias: the stacked all-kernel total df


[SIZE DIVERGENCE] all kernels all_kernels_0701
          kernel        host  np  size  nsampRatio ref_timer  n_timer  tm_div_sum_ns  tr_div_sum_ns  tm_div_mean_ns  tr_div_mean_ns  tm_div_pct_ref  tr_div_pct_ref  reduction  reduction_pct
          gs2d5p     c920bn3  64    64         0.5   cntvcto        4     311.247642       6.943643       77.811910        1.735911       21.888020        0.488301   0.977691      97.769094
          gs2d5p     c920bn3  64   128         0.5   cntvcto        4     293.229262      14.186804       73.307316        3.546701       10.273394        0.497040   0.951619      95.161873
          gs2d5p     c920bn3  64   256         0.5   cntvcto        4     307.749794       8.462781       76.937449        2.115695        5.435355        0.149466   0.972501      97.250110
          gs2d5p     c920bn3  64   512         0.5   cntvcto        4     309.907586      48.203476       77.476896       12.050869        2.734978        0.425402   0.844459      84.445855
   

In [32]:
total_div_df[['kernel','host','size','reduction_pct']]

wide = total_div_df[['kernel','host','size','reduction_pct']].pivot(
    index=["kernel", "size"],
    columns="host",
    values="reduction_pct"
).reset_index()

wide.columns.name = None

wide = wide.rename(columns={
    c: f"{c}(%)"
    for c in wide.columns
    if c not in ["kernel", "size"]
})

print(wide)
wide = wide[['kernel','size','cgnr6760pn2(%)','camd9554n1(%)','c920bn3(%)']]
kernel_list = list(wide['kernel'].unique())
for k in kernel_list:
    print(wide[wide['kernel'] == k].sort_values('size')[:4].to_string(index=False))

              kernel  size  c920bn3(%)  camd9554n1(%)  cgnr6760pn2(%)
0             gs2d5p    64   97.769094      96.817519       96.911154
1             gs2d5p   128   95.161873      98.022486       96.227439
2             gs2d5p   256   97.250110      97.929301       95.354236
3             gs2d5p   512   84.445855      97.151320       94.133114
4             gs2d5p  1024   56.348230      85.980933       85.849525
..               ...   ...         ...            ...             ...
65  tl_f90_cg_calc_w   128   93.965616      97.761783       97.605058
66  tl_f90_cg_calc_w   256   84.836437      95.056576       80.513535
67  tl_f90_cg_calc_w   512   29.236974      81.653444       76.783569
68  tl_f90_cg_calc_w  1024   20.203298      82.584680       87.431304
69  tl_f90_cg_calc_w  2048   16.970684      82.861153       68.658450

[70 rows x 5 columns]
kernel  size  cgnr6760pn2(%)  camd9554n1(%)  c920bn3(%)
gs2d5p    64       96.911154      96.817519   97.769094
gs2d5p   128       96.227

In [ ]:
# ---- LaTeX table: cross-timer Wasserstein divergence reduction (T_m -> T_r) ----
# Paper preamble needs: \usepackage{booktabs,multirow}. Reads `wide` (kernel,size,<host>(%)).
# ==== EASILY CONFIGURABLE ====
KERNEL_TEX = [                       # (data kernel id, paper display name) -> inclusion / order / name
    ('jacobi2d5p',       'Jacobi Stencil'),
    ('gs2d5p',           'Gauss-Seidel Stencil'),
    ('tl_f90_cg_calc_w', 'TeaLeaf CG Stencil'),
    ('openblas_gemm',    'GEMM'),
    ('openblas_gemv',    'GEMV'),
    ('hpcg_spmv',        'SpMV'),
    ('openblas_axpy',    'AXPY'),
    ('openblas_dot',     'DOT'),
    ('stream_triad',     'Triad'),
    ('npb_ft_fft',       'FFT'),
]
N_SHOW    = 2            # size rows shown per kernel
N_COMMENT = 2            # extra size rows kept but commented out (%) for easy manual toggling
SIZES_ASC = True         # take the smallest sizes first
TABLE_POS = 'htbp'
FONT_SIZE = r'\small'    # -> r'\footnotesize' if still too wide
TWO_COL   = True         # kernels split into left/right halves (fills the table* width)
N_LEFT    = 5            # number of kernels in the LEFT half
COL_GAP   = r'@{\hspace{2.5em}}'
SIZE_OVERRIDE = {}       # optional per-kernel size list, e.g. {'hpcg_spmv': [8, 12, 16, 24]}
HOST_TEX = [('cgnr6760pn2', 'Intel'),   # short header names; full names go in the caption
            ('camd9554n1',  'AMD'),
            ('c920bn3',     'Kunpeng')]
# =============================

def _fmt(v):
    return '--' if pd.isna(v) else f'{v:.1f}'

def _sizes_for(kid):
    if kid in SIZE_OVERRIDE:
        return list(SIZE_OVERRIDE[kid])
    ss = sorted(wide.loc[wide['kernel'] == kid, 'size'].unique(), reverse=not SIZES_ASC)
    return ss[:N_SHOW + N_COMMENT]

_hcols = [f'{h}(%)' for h, _ in HOST_TEX]
_ncell = 2 + len(HOST_TEX)          # kernel + size + hosts
_blank = ' & '.join([''] * _ncell)

def _kernel_rows(kid, name):
    """Return [(is_comment, cells_str)] for one kernel block; cells_str has no trailing '\\\\'."""
    if kid not in set(wide['kernel']):
        print('WARNING: kernel not in data (blank row):', kid)
        return [(False, _blank)]
    sub = wide[wide['kernel'] == kid].set_index('size')
    sizes = _sizes_for(kid)
    show, comm = sizes[:N_SHOW], sizes[N_SHOW:N_SHOW + N_COMMENT]
    rows = []
    for i, sz in enumerate(show):
        vals = ' & '.join(_fmt(sub.loc[sz, c]) for c in _hcols)
        lead = (r'\multirow{%d}{*}{%s}' % (len(show), name)) if i == 0 else ''
        rows.append((False, f'{lead} & {int(sz)} & {vals}'))
    for sz in comm:
        vals = ' & '.join(_fmt(sub.loc[sz, c]) for c in _hcols)
        rows.append((True, f' & {int(sz)} & {vals}'))
    return rows

_metric = r'\multicolumn{3}{c}{Cross-timer divergence reduction $T_m\!\rightarrow\!T_r$ (\%) $\uparrow$}'
_hlabels = ' & '.join(lab for _, lab in HOST_TEX)
_caption = (r'Cross-timer timing-measurement divergence reduction of the TacVar filter, per kernel and grid size. '
            r'Each entry is the percentage reduction in the aggregate Wasserstein distance between the non-reference timers '
            r'and the per-platform reference timer, from the raw measured time~$T_m$ to the filtered residual time~$T_r$ '
            r'(reference timer: TSC on x86, CNTVCT on ARM; $\mathrm{np}=64$). '
            r'Platforms: Intel Xeon 6760P, AMD EPYC 9554, Kunpeng 920B. '
            r'Higher is better~($\uparrow$): a larger value means the filter removes more cross-timer disagreement; '
            r'a negative value means the filter \emph{increases} divergence at that operating point.')

if TWO_COL:
    L = [_kernel_rows(k, n) for k, n in KERNEL_TEX[:N_LEFT]]
    R = [_kernel_rows(k, n) for k, n in KERNEL_TEX[N_LEFT:]]
    body = []
    for bi in range(max(len(L), len(R))):
        lb = L[bi] if bi < len(L) else None
        rb = R[bi] if bi < len(R) else None
        nrows = max(len(lb) if lb else 0, len(rb) if rb else 0)
        blk = []
        for ri in range(nrows):
            lc, lcells = lb[ri] if (lb and ri < len(lb)) else (True, _blank)
            rc, rcells = rb[ri] if (rb and ri < len(rb)) else (True, _blank)
            pre = '%  ' if (lc and rc) else ''      # only comment when BOTH halves are comment rows
            blk.append(f'{pre}{lcells} & {rcells} ' + r'\\')
        body.append('\n'.join(blk))
    tab_body = ('\n' + r'\midrule' + '\n').join(body)
    colspec = 'llrrr' + COL_GAP + 'llrrr'
    header = (r'\multirow{2}{*}{Kernel} & \multirow{2}{*}{Grid size} & ' + _metric + ' & '
              r'\multirow{2}{*}{Kernel} & \multirow{2}{*}{Grid size} & ' + _metric + r' \\' + '\n'
              r'\cmidrule(lr){3-5}\cmidrule(lr){8-10}' + '\n'
              f' & & {_hlabels} & & & {_hlabels} ' + r'\\')
else:
    body = []
    for k, n in KERNEL_TEX:
        body.append('\n'.join((('%  ' if c else '') + cells + r' \\') for c, cells in _kernel_rows(k, n)))
    tab_body = ('\n' + r'\midrule' + '\n').join(body)
    colspec = 'llrrr'
    header = (r'\multirow{2}{*}{Kernel} & \multirow{2}{*}{Grid size} & ' + _metric + r' \\' + '\n'
              r'\cmidrule(lr){3-5}' + '\n' + f' & & {_hlabels} ' + r'\\')

_lines = [r'\begin{table*}[%s]' % TABLE_POS, r'\centering', FONT_SIZE,
          r'\caption{%s}' % _caption, r'\label{tab:xtimer-divergence-reduction}',
          r'\begin{tabular}{%s}' % colspec, r'\toprule', header, r'\midrule',
          tab_body, r'\bottomrule', r'\end{tabular}', r'\end{table*}']
tex = '\n'.join(_lines)
print(tex)
if SAVE_FIG:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    _texpath = OUT_DIR / 'all_kernels_0701_wd_reduction.tex'
    _texpath.write_text(tex)
    print('\nsaved:', _texpath)


\begin{table*}[htbp]
\centering
\small
\caption{Cross-timer timing-measurement divergence reduction of the TacVar filter, per kernel and grid size. Each entry is the percentage reduction in the aggregate Wasserstein distance between the non-reference timers and the per-platform reference timer, from the raw measured time~$T_m$ to the filtered residual time~$T_r$ (reference timer: TSC on x86, CNTVCT on ARM; $\mathrm{np}=64$). Platforms: Intel Xeon 6760P, AMD EPYC 9554, Kunpeng 920B. Higher is better~($\uparrow$): a larger value means the filter removes more cross-timer disagreement; a negative value means the filter \emph{increases} divergence at that operating point.}
\label{tab:xtimer-divergence-reduction}
\begin{tabular}{llrrr@{\hspace{2.5em}}llrrr}
\toprule
\multirow{2}{*}{Kernel} & \multirow{2}{*}{Grid size} & \multicolumn{3}{c}{Cross-timer divergence reduction $T_m\!\rightarrow\!T_r$ (\%) $\uparrow$} & \multirow{2}{*}{Kernel} & \multirow{2}{*}{Grid size} & \multicolumn{3}{c}{Cross

In [34]:
# Coverage sanity (counts only; the deliverable is the fine-grained total_div_df above).
cov = (total_div_df.groupby('kernel')
       .agg(rows=('reduction_pct', 'size'),
            hosts=('host', 'nunique'),
            sizes=('size', 'nunique'))
       .reset_index())
print('kernels:', len(cov), '| total rows (kernel x host x size):', len(total_div_df))
print(cov.to_string(index=False))
print('nan reduction rows:', int(total_div_df['reduction_pct'].isna().sum()))


kernels: 12 | total rows (kernel x host x size): 210
          kernel  rows  hosts  sizes
          gs2d5p    18      3      6
       hpcg_spmv    21      3      7
      jacobi2d5p    18      3      6
          npb_ep    18      3      6
      npb_ft_fft    21      3      7
   openblas_axpy    18      3      6
    openblas_dot    18      3      6
   openblas_gemm    12      3      4
   openblas_gemv    18      3      6
  polybench_gemm    12      3      4
    stream_triad    18      3      6
tl_f90_cg_calc_w    18      3      6
nan reduction rows: 0
